In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import rasterio

generated_folder = "/home/ebr/projects/release-volume-sampler/generated"
region = "messina_002"
rasters_dir = os.path.join(generated_folder, region, "volumes/rasters")
bathy_path = os.path.join(generated_folder, region,  "bathy.tif")

# Read bathymetry
with rasterio.open(bathy_path) as src:
    bathy = src.read(1)
    bounds = src.bounds
    extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

plt.figure(figsize=(10, 8))
plt.imshow(bathy, cmap='Blues', extent=extent, origin='upper')
plt.title("Release Volumes as Contours over Bathymetry")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

# Plot all .tif rasters as contours
for fname in os.listdir(rasters_dir):
    if fname.endswith(".tif"):
        fpath = os.path.join(rasters_dir, fname)
        with rasterio.open(fpath) as src:
            data = src.read(1)
            bounds = src.bounds
            extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
            # Mask no-data values if needed
            data = np.ma.masked_equal(data, src.nodata)
            # Generate coordinates
            x = np.linspace(bounds.left, bounds.right, data.shape[1])
            y = np.linspace(bounds.bottom, bounds.top, data.shape[0])
            X, Y = np.meshgrid(x, y)
            # Plot contour for nonzero values
            if np.any(data > 0):
                plt.contour(X, Y, data, levels=[0], colors='r', linewidths=1, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = os.path.join(generated_folder, region, "volumes/volumes.csv")
df = pd.read_csv(csv_path)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['area'].dropna(), bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Area')
plt.ylabel('Count')
plt.title('Histogram of Area')

plt.subplot(1, 2, 2)
plt.hist(df['no2d'].dropna(), bins=30, color='salmon', edgecolor='black')
plt.xlabel('no2d')
plt.ylabel('Count')
plt.title('Histogram of no2d')

plt.tight_layout()
plt.show()

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

db_path = os.path.join(generated_folder, region, "volumes/volumes.db")

# Connect and read the area column
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query("SELECT area FROM volumes", conn)

plt.figure(figsize=(8, 5))
plt.hist(df['area'].dropna(), bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Area')
plt.ylabel('Count')
plt.title('Histogram of Area from volumes.db')
plt.tight_layout()
plt.show()

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query("SELECT area, condprob, p_fos_seed FROM volumes", conn)

weights = df['condprob'] * df['p_fos_seed']

plt.figure(figsize=(8, 5))
plt.hist(df['area'], bins=40, weights=weights, color='skyblue', edgecolor='black')
plt.xlabel('Area')
plt.ylabel('Weighted Count')
plt.title('Weighted Area Distribution (condprob * p_fos_seed)')
plt.tight_layout()
plt.show()

In [ ]:
import sqlite3
import numpy as np
import matplotlib.pyplot as plt

db_path = os.path.join(generated_folder, region, "volumes/volumes.db")
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query("SELECT area, condprob, p_fos_seed FROM volumes", conn)

areas = df['area'].values
probs = (df['condprob'] * df['p_fos_seed']).values

a_grid = np.linspace(np.nanmin(areas), np.nanmax(areas), 200)
prob_curve = []

for a in a_grid:
    mask = areas > a
    # Probability that none of the volumes with area > a occur
    p_none = np.prod(1 - probs[mask])
    # Probability that at least one volume with area > a occurs
    prob_curve.append(1 - p_none)


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(a_grid, prob_curve, color='navy')
plt.xlabel('Area threshold (a)')
plt.ylabel('Probability(area > a)')
plt.yscale('log')
plt.title('Probability Curve: area > a')
plt.grid(True)
plt.tight_layout()
plt.show()

# Triangulation Class Test

This notebook demonstrates how to import and use the `Triangulation` class.

In [ ]:
# Import sys and os to allow importing from src/rvsampler
import sys
import os
import numpy as np
import matplotlib.pyplot as plt


# Add the src directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now import the Triangulation class
from rvsampler.triangulate import Triangulation
from rvsampler.utils import create_dir, read_tif, write_tif, write_content 

In [ ]:
rundir = "/home/ebr/projects/release-volume-sampler/generated/messina_001"
triang = Triangulation(rundir)
triang.initialize_triangle_properties()

You can now create and use a `Triangulation` object as needed.

In [ ]:
triang.slopeunits[0]

In [ ]:
plt.imshow(triang.tri_mask)


In [ ]:

triangles_from_slopeunits = {}
triangle_indices = np.arange(triang.n_triangles)
for slopeunit in set(triang.slopeunits):
    print(slopeunit)
    triangles_from_slopeunits[slopeunit] = triangle_indices[triang.slopeunits == slopeunit]

In [ ]:

triang.slopeunits_to_triangles(filename="slopeunits_to_triangles.npy")

In [ ]:

slopeunits_to_triangles = np.load(os.path.join(rundir, "triangulation", "slopeunits_to_triangles.npy"), allow_pickle=True).item()

In [ ]:
slopeunits_to_triangles = np.load(os.path.join(rundir, "triangulation", "slopeunits_to_triangles.npy"))

In [ ]:
slopeunits_to_triangles[1]

In [ ]:
triangles_from_slopeunits[9]

In [ ]:
triangles_from_slopeunits[3]

In [ ]:
import rasterio
import os

tri_mask_path = os.path.join(rundir,"triangulation", "triangulation.tif")
with rasterio.open(tri_mask_path) as src:
    tri_mask = src.read(1)
    tri_profile = src.profile
    bounds = src.bounds
    crs = src.crs
    transform = src.transform

In [ ]:
slopeunit_mask = np.zeros(tri_mask.shape)
triangles_from_slopeunits = {}

for slopeunit in set(slopeunits):
    triangles_in_slopeunit = triangle_indices[slopeunits == slopeunit]
    slopeunit_mask[np.isin(tri_mask, triangles_in_slopeunit)] = slopeunit
    triangles_from_slopeunits[slopeunit] = triangles_in_slopeunit
#np.isin(tri_mask, triangles_from_slopeunits[8])


In [ ]:
slopeunit_mask[slopeunit_mask > 2000] = np.nan

In [ ]:
plt.imshow(slopeunit_mask)

In [ ]:
slopeunitfile = os.path.join(rundir, "slumap.tif")
tri_mask_path = os.path.join(rundir, "triangulation", "triangulation.tif")

In [ ]:
triangles_raster, _, triangles_raster_mask_profile = read_tif(tri_mask_path)
slopeunits_raster, slopeunits_raster_mask, slopeunits_mask_profile = read_tif(slopeunitfile)

In [ ]:
triangles_raster = triangles_raster.astype(int)
slopeunits_raster = slopeunits_raster.astype(int)

In [ ]:
slopeunits_raster[slopeunits_raster > 2000] = np.


In [ ]:
plt.imshow(slopeunits_raster)

In [ ]:
plt.imshow(triangles_raster)

In [ ]:
from scipy import ndimage

slopeunits = ndimage.labeled_comprehension(
                slopeunits_raster, triangles_raster, np.arange(triang.n_triangles),
                lambda x: np.bincount(x).argmax(), float, np.nan
            ).astype(int)

In [ ]:
slopeunits.shape

In [ ]:

def cluster_probabilities(probabilities, volumes_file, cluster_file, resdir, bath_file):
    df_full = pd.read_csv(volumes_file)
    df_cluster = pd.read_csv(cluster_file)
    
    # This method uses only the volumes with single seed!!!!!!!!!!#######
    # For the other method se below
    # Unique clusters and seeds from the full dataset
    all_clusters = np.arange(df_full['cluster'].max() + 1)
    all_seeds = np.unique(df_full['seed_triangle'])

    # Filter only rows with seed_triangle2 == -1
    filtered = df_full[df_full['seed_triangle2'] == -1]

    # Group by cluster and seed_triangle, then sum condprob
    grouped = filtered.groupby(['cluster', 'seed_triangle'])['condprob'].sum().reset_index()

    # Pivot to 2D array, then reindex to ensure full dimensions
    pivot = grouped.pivot(index='cluster', columns='seed_triangle', values='condprob')

    # Reindex to fill in missing clusters and seeds with zeros
    pivot = pivot.reindex(index=all_clusters, columns=all_seeds, fill_value=0)

    # Convert to NumPy array
    all_probs = pivot.to_numpy()
    cluster_ids = pivot.index.to_numpy()
    seed_ids = pivot.columns.to_numpy()
    all_probs2 = all_probs * probabilities[seed_ids][np.newaxis, :]

    df_cluster['prob1'] = np.nansum(all_probs2,axis=1)
    
    # This method assumes that all unique seed triangle combinations are their own tree
    # Pre-allocate result array
    Pc2 = np.zeros(500)

    # Extract only needed columns as arrays for speed
    seed1 = df_full['seed_triangle'].values
    seed2 = df_full['seed_triangle2'].values
    cluster = df_full['cluster'].values
    condprob = df_full['condprob'].values

    # Loop through clusters
    for cin in range(500):
        # Filter rows for this cluster
        mask = cluster == cin
        seed_matrix = np.column_stack((seed1[mask], seed2[mask]))
        condprob_c = condprob[mask]

        # Get unique combinations and inverse indices
        unique_combinations, inverse_idx = np.unique(seed_matrix, axis=0, return_inverse=True)

        # Sum condprob for each unique seed pair
        prob_sums = np.zeros(len(unique_combinations))
        np.add.at(prob_sums, inverse_idx, condprob_c)

        # Compute weighted contribution
        for i, (s1, s2) in enumerate(unique_combinations):
            p1 = probabilities[s1]
            p2 = probabilities[s2] if s2 > -1 else 1
            Pc2[cin] += prob_sums[i] * p1 * p2
    df_cluster['prob2'] = Pc2
    
    # Egentlig cluster_file, men siden jeg driver å tester trenger jeg ikke skrive over foreløpig
    cluster_file2 = os.path.join(resdir, 'volumes','Clusters2.csv')
    df_cluster.to_csv(cluster_file2, index=False)
    plot_probabilities(df_full,resdir, probabilities, bath_file)

In [ ]:
def plot_probabilities(df_this,resdir, probabilities, bath_file):
    # Flatten and count seeds (excluding -1)
    allseeds_flat = pd.concat([df_this['seed_triangle'], df_this['seed_triangle2']])
    allseeds_flat = allseeds_flat[allseeds_flat != -1]
    seed_counts = allseeds_flat.value_counts()

    # Map counts back to each row (get max of both seed columns)
    def get_row_count(row):
        seeds = [row['seed_triangle'], row['seed_triangle2']]
        return max(seed_counts.get(seeds[0], 0), seed_counts.get(seeds[1], 0))

    # Compute frequencies per row
    df_this['seed_count'] = df_this.apply(get_row_count, axis=1)

    # Filter valid rows (exclude -1)
    valid_mask = (df_this['seed_triangle'] != -1)

    df_this['seed_prob'] = probabilities[df_this['seed_triangle']]
    # 1. Get unique seed_triangle values (excluding -1 if needed)
    unique_seeds = df_this['seed_triangle'].unique()
    unique_seeds = unique_seeds[unique_seeds != -1]

    # 2. For each unique seed, get the first occurrence’s coordinates
    grouped = df_this[df_this['seed_triangle'].isin(unique_seeds)].groupby('seed_triangle').first()

    # 3. Get longitude, latitude, and corresponding probability
    lons = grouped['lon']
    lats = grouped['lat']
    probs = probabilities[grouped.index]

    valid_mask = (
        df_this['seed_count'].notna() &
        df_this['lon'].notna() &
        df_this['lat'].notna()
    )

    # Apply mask to all 3 series
    lons2 = df_this.loc[valid_mask, 'lon']
    lats2 = df_this.loc[valid_mask, 'lat']
    weights2 = df_this.loc[valid_mask, 'seed_count']

    # Load the batymetri
    with rasterio.open(bath_file) as src:
        bathymetri = src.read(1)  # First band
        bathymetri[bathymetri > 60000] = 0
        bounds = src.bounds 
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]


    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # First plot: seed count with logarithmic color scale
    im0 = axes[0].imshow(bathymetri, cmap='grey', extent=extent, origin='upper')
    _, _, _, sc0 = axes[0].hist2d(
        lons2, lats2,
        bins=100,
        cmap='tab20b',
        weights=weights2,
        norm=LogNorm(vmin=1, vmax=weights2.max())  # adjust as needed
    )
    fig.colorbar(sc0, ax=axes[0], label='Volume count')
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].set_title('Volume locations')
    axes[0].grid(True)

    # Create masked probabilities
    probs_masked = np.where(probs > 0, probs, np.nan)

    # Plot bathymetri background
    im1 = axes[1].imshow(bathymetri, cmap='gray', extent=extent, origin='upper')

    valid_mask = (
        ~np.isnan(lats) &
        ~np.isnan(lons) &
        ~np.isnan(probs_masked)
    )

    # Apply the mask
    lats_valid = lats[valid_mask]
    lons_valid = lons[valid_mask]
    probs_valid = probs_masked[valid_mask]

    # Compute 2D histogram manually
    H, xedges, yedges = np.histogram2d(
        lats_valid, lons_valid,
        bins=100,
        weights=probs_valid
    )

    # Mask where H is zero or NaN
    H_masked = np.ma.masked_where((H == 0) | np.isnan(H), H)

    # Define meshgrid for pcolormesh
    X, Y = np.meshgrid(yedges, xedges)

    # Overlay histogram using pcolormesh (respects masking)
    pc = axes[1].pcolormesh(X, Y, H_masked, cmap='tab20b', norm=Normalize(vmin=0.001, vmax=0.3))

    # Add colorbar and labels
    fig.colorbar(pc, ax=axes[1], label='Probability')
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    axes[1].set_title('Probability from shakemap')
    axes[1].grid(True)

    plt.tight_layout()
    filename = os.path.join(resdir,'volumes','Volumes_probs_new.png')
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()